MicroTorch Example

Simulates data from the ball-stick model


In [1]:
import numpy as np
import torch
import sys
sys.path.append('../data/')
sys.path.append('../')

In [2]:
from load_data import load_grad

In [4]:
grad = load_grad('../data/grad_files/schemefile_hcp_wu_minn.scheme')
grad2 = load_grad('../data/grad_files/grad_HCP.txt')
#convert to microns^2/ms
grad[:,:4] = grad2
import numpy as np
np.savetxt('../data/grad_files/grad_HCP_microns.txt', grad, fmt='%1.4e')


In [ ]:
def convert_comp_strings_to_classes(comps_strings):
    import importlib
    signal_models_module = importlib.import_module("signal_models")

    comps_classes = () #initialise tuple
    for comp in comps_strings:
        #get the class
        this_class = getattr(signal_models_module, comp) #add to the tuple
        #create an instance of the class and add to the tuple
        comps_classes += (this_class(),)
    return comps_classes

In [ ]:
#from signal_models import Ball 
#from signal_models import Stick


from model_maker import ModelMaker

comps = ("NEXI",)
modelfunc = ModelMaker(convert_comp_strings_to_classes(comps))

comps = ("Ball", )
ball_modelfunc = ModelMaker(convert_comp_strings_to_classes(comps))

comps = ("Stick", )
stick_modelfunc = ModelMaker(convert_comp_strings_to_classes(comps))




In [ ]:
modelfunc.n_params
modelfunc.n_frac

In [ ]:
import numpy as np
import torch

def generate_random_params(modelfunc, repeat_interval=1):
    """
    Generate random parameters with an option to repeat every `repeat_interval` elements in a row.
    
    Args:
        modelfunc: The model function with `parameter_ranges` and `n_frac` attributes.
        repeat_interval: Number of elements to repeat. Default is 1 (no repetition).
        
    Returns:
        params: Tensor of shape [1000, modelfunc.n_params + modelfunc.n_frac] with random parameters.
    """
    # Extract min and max values from the columns
    min_vals = modelfunc.parameter_ranges[:, 0]
    max_vals = modelfunc.parameter_ranges[:, 1]

    # Add the volume fraction min/max values
    min_vals = np.append(min_vals, np.ones(modelfunc.n_frac))
    max_vals = np.append(max_vals, np.zeros(modelfunc.n_frac))

    # Calculate the number of unique sets needed
    n_unique = 1000 // repeat_interval

    # Generate n_unique sets of parameters with random values within the min and max ranges
    unique_params = (torch.rand(n_unique, modelfunc.n_params + modelfunc.n_frac) * (max_vals - min_vals) + min_vals).float()

    # Repeat each set of parameters `repeat_interval` times
    params = unique_params.repeat_interleave(repeat_interval, dim=0)

    return params

# Example usage
params = generate_random_params(modelfunc, repeat_interval=10)
# params_ball = generate_random_params(ball_modelfunc, repeat_interval=10)
# params_stick = generate_random_params(stick_modelfunc, repeat_interval=10)

S = modelfunc(grad, params)
# S_ball = ball_modelfunc(grad, params_ball)
# S_stick = stick_modelfunc(grad, params_stick)

# Reshape the signal tensor and parameters tensor into the desired image format
Simg = S.view(20, 10, 5, 288)
paramsimg = params.view(20, 10, 5, params.size(-1))
mask = torch.ones_like(Simg[:, :, :, 0])

#save the image using nibabel
import nibabel as nib
img = nib.Nifti1Image(Simg.numpy(), np.eye(4))
nib.save(img, '../data/test_images/' + ''.join(modelfunc.comp_names) + '.nii.gz')

maskimg = nib.Nifti1Image(mask.numpy(), np.eye(4))
# nib.save(img)
nib.save(maskimg, '../data/test_images/nexi_mask.nii.gz')




In [ ]:
S.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(paramsimg[0,:, :,1])
plt.colorbar()
#plt.plot(grad[:,3], Simg[0,0,0,:],'o')

In [ ]:
f = 0.3
Dpar = 2 
D = 3
theta = 0
phi = np.pi

ball_params = torch.tensor([D], dtype=torch.float32)
ball_params = torch.unsqueeze(ball_params, dim=0)

stick_params = torch.tensor([Dpar, theta, phi], dtype=torch.float32)
stick_params = torch.unsqueeze(stick_params, dim=0)

f_params = torch.tensor([f], dtype=torch.float32)
f_params = torch.unsqueeze(f_params, dim=0)

# Concatenate along dimension 1 (columns)
params = torch.cat([ball_params, stick_params, f_params], dim=1)

print(params.type())

S = modelfunc(params,grad)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(grad[:,3],S[170,:],'o')

In [ ]:
grad

In [ ]:
paramsimg.size()

In [ ]:
plt.imshow(paramsimg[:,:, 0, 3])